# SED B3 Freq-Attention + Multi-Window Inference

マルチウィンドウ推論: 通常窓 + 2.5秒ずらし窓の予測を平均してFNを削減。

| Setting | Value |
|---------|-------|
| Accelerator | None (CPU) |
| Internet | OFF |

In [ ]:
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
import torchaudio.transforms as T
import os, time, glob
from pathlib import Path
from dataclasses import dataclass
from concurrent.futures import ThreadPoolExecutor

START = time.time()
print(f'PyTorch {torch.__version__}')

# CONFIG
@dataclass
class Config:
    sr: int = 32_000
    chunk_duration: float = 5.0
    n_mels: int = 224
    n_fft: int = 2048
    hop_length: int = 512
    fmin: int = 0
    fmax: int = 16_000
    top_db: float = 80.0
    power: float = 2.0
    norm: str = 'slaney'
    mel_scale: str = 'htk'
    backbone: str = 'tf_efficientnet_b3.ns_jft_in1k'
    num_classes: int = 234
    in_channels: int = 3
    dropout: float = 0.1
    drop_path_rate: float = 0.0
    gem_p_init: float = 3.0
    @property
    def chunk_samples(self): return int(self.sr * self.chunk_duration)

cfg = Config()

# PATHS
DATA_ROOT = None
for p in ['/kaggle/input/birdclef-2026', '/kaggle/input/competitions/birdclef-2026']:
    if os.path.exists(p): DATA_ROOT = p; break
assert DATA_ROOT, 'Competition data not found'

CHECKPOINT = None
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        if f.endswith('.pt'):
            CHECKPOINT = os.path.join(root, f)
            break
    if CHECKPOINT:
        break
assert CHECKPOINT, '.pt file not found'
print(f'Checkpoint: {CHECKPOINT}')

TEST_DIR = os.path.join(DATA_ROOT, 'test_soundscapes')
TRAIN_SC_DIR = os.path.join(DATA_ROOT, 'train_soundscapes')
sub_df = pd.read_csv(os.path.join(DATA_ROOT, 'sample_submission.csv'), nrows=1)
SPECIES = list(sub_df.columns[1:])
print(f'Species: {len(SPECIES)}')

In [ ]:
# MODEL
class GEMFreqPool(nn.Module):
    def __init__(self, p_init=3.0, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.tensor(p_init))
        self.eps = eps
    def forward(self, x):
        with torch.amp.autocast('cuda', enabled=False):
            x = x.float()
            p = self.p.clamp(min=1.0)
            x = x.clamp(min=self.eps).pow(p)
            x = x.mean(dim=2)
            x = x.pow(1.0 / p)
        return x

class AttentionSEDHead(nn.Module):
    def __init__(self, feat_dim, num_classes, dropout=0.1):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(feat_dim, feat_dim), nn.ReLU(inplace=True), nn.Dropout(dropout))
        self.att_conv = nn.Conv1d(feat_dim, num_classes, kernel_size=1)
        self.cls_conv = nn.Conv1d(feat_dim, num_classes, kernel_size=1)
    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.fc(x)
        x = x.permute(0, 2, 1)
        att = torch.tanh(self.att_conv(x))
        att = F.softmax(att, dim=-1)
        cls = self.cls_conv(x)
        clipwise_logit = (att * cls).sum(dim=-1)
        return {
            'clipwise_prob': torch.sigmoid(clipwise_logit),
            'segmentwise_logit': cls.permute(0, 2, 1),
        }

class SEDModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.backbone = timm.create_model(
            cfg.backbone, pretrained=False,
            in_chans=cfg.in_channels, features_only=False,
            global_pool='', num_classes=0,
            drop_path_rate=cfg.drop_path_rate)
        self.gem_pool = GEMFreqPool(p_init=cfg.gem_p_init)
        self.head = AttentionSEDHead(self.backbone.num_features, cfg.num_classes, cfg.dropout)
    def forward(self, x):
        return self.head(self.gem_pool(self.backbone(x)))

# MEL TRANSFORM - 転置版 (周波数方向Attention)
class MelSpectrogramTransform(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.mel = T.MelSpectrogram(
            sample_rate=cfg.sr, n_fft=cfg.n_fft, hop_length=cfg.hop_length,
            n_mels=cfg.n_mels, f_min=cfg.fmin, f_max=cfg.fmax,
            power=cfg.power, norm=cfg.norm, mel_scale=cfg.mel_scale)
        self.db = T.AmplitudeToDB(stype='power', top_db=cfg.top_db)
    @torch.no_grad()
    def forward(self, waveforms):
        with torch.amp.autocast('cuda', enabled=False):
            waveforms = waveforms.float()
            mel = self.db(self.mel(waveforms))
            B = mel.shape[0]
            flat = mel.reshape(B, -1)
            mn = flat.min(dim=1, keepdim=True)[0].unsqueeze(-1)
            mx = flat.max(dim=1, keepdim=True)[0].unsqueeze(-1)
            mel = (mel - mn) / (mx - mn + 1e-7)
            mel = mel.unsqueeze(1).repeat(1, 3, 1, 1)
            # 周波数と時間を転置 → 「どの周波数帯で鳴いたか」を判定
            mel = mel.transpose(-2, -1)
        return mel

print('Model + Mel defined (freq-attention)')

In [ ]:
# LOAD MODEL
model = SEDModel(cfg)
ckpt = torch.load(CHECKPOINT, map_location='cpu', weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
epoch = ckpt.get('epoch', '?')
val_auc = ckpt.get('metrics', {}).get('macro_auc', '?')
print(f'Model loaded: epoch={epoch}, val_auc={val_auc}')

mel_transform = MelSpectrogramTransform(cfg)
mel_transform.eval()

# FIND TEST FILES
test_files = sorted(glob.glob(os.path.join(TEST_DIR, '*.ogg')))
if not test_files:
    print('No test files, using train_soundscapes as fallback')
    test_files = sorted(glob.glob(os.path.join(TRAIN_SC_DIR, '*.ogg')))[:8]
print(f'Test files: {len(test_files)}')

In [ ]:
# AUDIO LOADING + MULTI-WINDOW INFERENCE
from collections import defaultdict

def load_soundscape(path):
    stem = Path(path).stem
    y, _ = librosa.load(path, sr=cfg.sr, mono=True)
    return y, stem

print('Loading audio files...')
t0 = time.time()
with ThreadPoolExecutor(max_workers=4) as executor:
    audio_results = list(executor.map(load_soundscape, test_files))
print(f'  Loaded {len(audio_results)} files in {time.time()-t0:.1f}s')

CHUNK = cfg.chunk_samples
HALF_CHUNK = CHUNK // 2  # 2.5秒分のサンプル数
BUCKET_SEC = int(cfg.chunk_duration)  # 5秒

def predict_windows(audio, start_samples, model, mel_transform):
    """指定した開始位置群から5秒窓を切り出してバッチ推論"""
    windows = []
    for s in start_samples:
        e = s + CHUNK
        w = audio[s:e]
        if len(w) < CHUNK:
            w = np.pad(w, (0, CHUNK - len(w)))
        windows.append(w)
    if not windows:
        return []
    chunks_tensor = torch.from_numpy(np.stack(windows)).float()
    with torch.no_grad():
        mel = mel_transform(chunks_tensor)
        probs = model(mel)['clipwise_prob'].numpy()
    return probs

# マルチウィンドウ推論: 通常窓 + 2.5秒ずらし窓
# 各バケット(5秒区間)に重なる窓の予測を平均
bucket_preds = {}  # row_id -> list of predictions

print('Running multi-window inference...')
t0 = time.time()
for audio, stem in audio_results:
    # ピーク正規化
    peak = np.abs(audio).max()
    if peak > 0:
        audio = audio / peak

    n_chunks = max(1, len(audio) // CHUNK)
    total_len = n_chunks * CHUNK
    if len(audio) < total_len:
        audio = np.pad(audio, (0, total_len - len(audio)))
    else:
        audio = audio[:total_len]

    # 窓A: 通常窓 (0, 5, 10, ...)
    starts_a = [i * CHUNK for i in range(n_chunks)]
    probs_a = predict_windows(audio, starts_a, model, mel_transform)

    # 窓B: 2.5秒ずらし (2.5, 7.5, 12.5, ...)
    starts_b = [HALF_CHUNK + i * CHUNK for i in range(n_chunks) if HALF_CHUNK + i * CHUNK + CHUNK <= len(audio)]
    probs_b = predict_windows(audio, starts_b, model, mel_transform)

    # 各バケットに予測を集約
    for i in range(n_chunks):
        end_sec = (i + 1) * BUCKET_SEC
        row_id = f'{stem}_{end_sec}'
        if row_id not in bucket_preds:
            bucket_preds[row_id] = []
        # 窓Aの予測を追加
        if i < len(probs_a):
            bucket_preds[row_id].append(probs_a[i])
        # 窓Bの予測を追加（このバケットに重なるもの）
        # 窓B[i]は (i*5+2.5) ~ (i*5+7.5) なのでバケット i と i+1 に重なる
        if i < len(probs_b):
            bucket_preds[row_id].append(probs_b[i])  # 後半がこのバケットに重なる
        if i > 0 and (i-1) < len(probs_b):
            bucket_preds[row_id].append(probs_b[i-1])  # 前の窓Bの前半がこのバケットに重なる

# 平均化
all_row_ids = []
all_preds = []
for row_id, preds_list in bucket_preds.items():
    all_row_ids.append(row_id)
    all_preds.append(np.mean(preds_list, axis=0))

elapsed = time.time() - t0
print(f'  Multi-window inference done: {len(all_row_ids)} predictions in {elapsed:.1f}s')

In [ ]:
# BUILD SUBMISSION
preds_array = np.stack(all_preds)
submission = pd.DataFrame(preds_array, columns=SPECIES)
submission.insert(0, 'row_id', all_row_ids)

sample_sub = pd.read_csv(os.path.join(DATA_ROOT, 'sample_submission.csv'))
expected_ids = set(sample_sub['row_id'])
our_ids = set(submission['row_id'])

missing = expected_ids - our_ids
if missing:
    print(f'WARNING: {len(missing)} missing row_ids')
    missing_df = pd.DataFrame({'row_id': list(missing)})
    for sp in SPECIES:
        missing_df[sp] = 0.0
    submission = pd.concat([submission, missing_df], ignore_index=True)

extra = our_ids - expected_ids
if extra:
    print(f'Dropping {len(extra)} extra row_ids')
    submission = submission[submission['row_id'].isin(expected_ids)]

submission = submission.set_index('row_id').loc[sample_sub['row_id']].reset_index()
submission.to_csv('submission.csv', index=False)

total_time = time.time() - START
print(f'Submission saved: {submission.shape}')
print(f'Total time: {total_time:.0f}s ({total_time/60:.1f} min)')
print(submission.head())